# Testes de visualização 2D e 3D

Aluno: André Destefani Stefanato

Orientador: Roberto Colistete Jr

Importando as bibliotecas

In [5]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import pandas as pd
import os
!pip install astroquery
from astroquery.jplhorizons import Horizons
import math
from astropy.time import Time

Inicializando parte dos atributos

In [195]:
espacamento = 10
passos = 10000 # Passos da simulação (Na parte de visualização é usada como o total de frames)
dt = 0.0001
G = 4 * np.pi**2
epsilon = 0.001

In [196]:
tempoAlvo = Time('2026-07-24T20:00:00', format='isot', scale='utc')

#Ids dos corpos no Horizons
BODIES = {
    'Sol': 10,
    'Mercúrio': 199,
    'Vênus': 299,
    'Terra': 399,
    'Marte': 499,
    'Júpiter': 599,
    'Saturno': 699,
    'Urano': 799,
    'Netuno': 899
}

#Massas relativas ao Sol
MASSAS_SOLARES = {
    'Sol': 1.0,
    'Mercúrio': 1.660e-7,
    'Vênus': 2.447e-6,
    'Terra': 3.003e-6,
    'Marte': 3.227e-7,
    'Júpiter': 9.546e-4,
    'Saturno': 2.858e-4,
    'Urano': 4.366e-5,
    'Netuno': 5.151e-5
}


Pegando os dados dos planetas do JPL Horizons:

In [197]:
posicoes = [] # em AU
velocidades = [] # em AU/ano
massas = []
nomes = []

for nome, objId in BODIES.items():
  obj = Horizons(id=objId, location='500@10', epochs = tempoAlvo.jd)
  vec = obj.vectors()

  #Para as posições em AU
  pos = np.array([vec['x'][0], vec['y'][0], vec['z'][0]])

  #Para a velocidade
  vel = np.array([vec['vx'][0], vec['vy'][0], vec['vz'][0]]) * 365.25

  posicoes.append(pos)
  velocidades.append(vel)
  massas.append(MASSAS_SOLARES[nome])
  nomes.append(nome)

posicoes = np.array(posicoes)
velocidades = np.array(velocidades)
massas = np.array(massas)

df = pd.DataFrame({
    'x': posicoes[:, 0],
    'y': posicoes[:, 1],
    'z': posicoes[:, 2],
    'vx': velocidades[:, 0],
    'vy': velocidades[:, 1],
    'vz': velocidades[:, 2],
    'massa': massas
})

df['nome'] = nomes

df.to_hdf("Entrada.h5", key="Entrada", mode="w", format='table')

In [198]:
dados = pd.read_hdf("Entrada.h5")

Função de salvar snapshot e salvar em HDF5 feita pelo Raphael

In [199]:
def salvarSnapshot(df, biblioteca, dimensao):
  df.to_hdf("SnapshotTeste.h5", key=biblioteca + "/" + dimensao + "/Snapshots", mode="w", append=True, format='table' )

In [200]:
def salvarHDF5(kInicial, uInicial, kFinal, uFinal, biblioteca, dimensao):
  df = pd.DataFrame({"EnergiaCineticaInicial":[kInicial], "EnergiaPotencialInicial":[uInicial], "EnergiaCineticaFinal":[kFinal], "EnergiaPotencialFinal":[uFinal]})
  df.to_hdf("SaidaTeste.h5", key=biblioteca + "/" + dimensao + "/Resultados", mode="a", append=True)

## Visualização da simulação Numpy 2D e 3D

### <font color="red">2D</font>

Inicialização dos atributos feita em Python puro, essa parte do código sera executada apenas uma vez. Os vetores gerados serão jogados para o Numpy depois

In [ ]:
N = 64
numCorpos = N**2

m = [1] * (N**2)
vx = [0] * (N**2)
vy = [0] * (N**2)
x = []
y = []

x_temp = 0

for i in range(N):
  x_temp += espacamento
  y_temp = 0
  for j in range(N):
    y_temp += espacamento
    x.append(x_temp)
    y.append(y_temp)

Aqui jogamos os arrays criados com Python puro para o Numpy

In [ ]:
np_x = np.array(x, dtype=np.float64)
np_y = np.array(y, dtype=np.float64)
np_m = np.array(m, dtype=np.float64)
np_vx = np.array(vx, dtype=np.float64)
np_vy = np.array(vy, dtype=np.float64)

Função de cálculo da força gravitacional de cada partícula dentro do sistema. A função retorna a força x e y total para cada partícula

In [ ]:
def calcular_forcas(np_x, np_y, np_m, G, epsilon):
    dx = np_x[None, :] - np_x[:, None]
    dy = np_y[None, :] - np_y[:, None]

    dist = np.sqrt(dx**2 + dy**2 + epsilon**2)

    np.fill_diagonal(dist, np.inf)

    F = G * (np_m[:, None] * np_m[None, :]) / (dist**2)

    fx = F * (dx / dist)
    fy = F * (dy / dist)

    fx_total = np.sum(fx, axis=1)
    fy_total = np.sum(fy, axis=1)

    return fx_total, fy_total

Versão sem for, menos precisão mas muito mais rápida

In [ ]:
def atualizar_posicao(np_x, np_y, np_vx, np_vy, np_m, dt, G, epsilon):
    fx, fy = calcular_forcas(np_x, np_y, np_m, G, epsilon)

    ax = fx / np_m
    ay = fy / np_m

    np_vx += ax * dt
    np_vy += ay * dt

    np_x += np_vx * dt
    np_y += np_vy * dt

Função que calcula a energia cinética total do sistema

In [ ]:
def calcular_energia_cinetica(np_vx, np_vy, np_m):
    return 0.5 * np.sum(np_m * (np_vx**2 + np_vy**2))

Função que calcula a energia potencial total do sistema

In [ ]:
def calcular_energia_potencial(np_x, np_y, np_m, G, epsilon):
    dx = np_x[None, :] - np_x[: ,None]
    dy = np_y[None, :] - np_y[: ,None]

    dist = np.sqrt(dx**2 + dy**2 + epsilon**2)

    np.fill_diagonal(dist, np.inf)

    m_prod = np_m[:, None] * np_m[None, :]

    U = G * np.sum(m_prod / dist) / 2

    return U

---

In [ ]:
snapshots = np.empty([passos * numCorpos, 8], dtype=np.float64) # inicializando as snapshots vazias para depois serem preenchidas

for i in range(passos): # Simulação e inserção dos resultados de cada passo nas snapshots
    inicio = i * numCorpos # "Index" inicial pra esse passo
    fim = inicio + numCorpos # "Index" final pra esse passo

    snapshots[inicio:fim, 0:8] = np.array([np_x.copy(),
                                           np_y.copy(),
                                           np.full(numCorpos, 0),
                                           np_vx.copy(),
                                           np_vy.copy(),
                                           np.full(numCorpos, 0),
                                           np_m.copy(),
                                           np.full(numCorpos, i)]).transpose() # Salva os resultados atuais no snapshots

    atualizar_posicao(np_x, np_y, np_vx, np_vy, np_m, dt, G, epsilon) # Simula

df = pd.DataFrame(snapshots, columns=["x", "y", "z", "vx", "vy", "vz", "m", "passo"]) # Transforma o snapshots em uma DataFrame

salvarSnapshot(df, biblioteca="Numpy", dimensao="D2") # Salva o DataFrame com a função salvarSnapshot

df = pd.read_hdf("SnapshotTeste.h5") # Lê o arquivo

posicoes = df[["x", "y"]].values  # Pega as posições

velocidades = df[["vx", "vy"]].values # Pega as velocidade

fig, ax = plt.subplots() # Cria a figura e os pontos

ax.set_aspect("equal") # Evita que haja distorção entre os eixos x e y

plt.axis("off") # Remove os eixos

fig.set_facecolor("k") # Pinta o fundo da figura de preto

velocidades = np.sqrt(velocidades[:, 0]**2 + velocidades[:, 1]**2) # Calcula a velocidade de cada ponto

scat= ax.scatter([], [], c=[], cmap="bwr", s=0.8, vmin=velocidades.min(), vmax=velocidades.max()) # Função que insere os pontos

def update(frame): # Função update
    inicio = frame * numCorpos
    fim = inicio + numCorpos

    pos = posicoes[inicio:fim]
    vel = velocidades[inicio:fim]

    x = pos[:, 0]
    y = pos[:, 1]

    scat.set_array(vel)
    scat.set_offsets(pos)

    ax.set_xlim(x.min() - 10, x.max() + 10)
    ax.set_ylim(y.min() - 10, y.max() + 10)

    return scat,

anim = animation.FuncAnimation(fig, update, frames=passos, blit=False)

print("Renderizando vídeo...")

writer = animation.FFMpegWriter(fps=20, codec='libx264', bitrate=2000) # Define como a animação vai ser lida e como vai ser salva (20 fps, em .mp4)

anim.save("simulacao_corpos_2D.mp4", writer=writer) # Salva o animação um arquivo .mp4

plt.close() # Evita o plot estático inicial

print("O arquivo foi salvo com sucesso.")

Renderizando vídeo...
O arquivo foi salvo com sucesso.


### <font color="red">3D</font>

Para inicializar usando os dados da Horizons:

In [212]:
x = dados["x"].to_numpy().copy()
y = dados["y"].to_numpy().copy()
z = dados["z"].to_numpy().copy()
vx = dados["vx"].to_numpy().copy()
vy = dados["vy"].to_numpy().copy()
vz = dados["vz"].to_numpy().copy()
m = dados["massa"].to_numpy().copy()

numCorpos = 9

Para inicializar em forma de um cubo:

In [194]:
N = 64

m = [1] * (N**3)
vx = [0] * (N**3)
vy = [0] * (N**3)
vz = [0] * (N**3)
x = []
y = []
z = []

x_temp = 0

for i in range(N):
  x_temp += espacamento
  y_temp = 0
  for j in range(N):
    y_temp += espacamento
    z_temp = 0
    for k in range(N):
      z_temp += espacamento
      x.append(x_temp)
      y.append(y_temp)
      z.append(z_temp)

Jogando esses atributos para o numpy:

In [213]:
np_x = np.array(x, dtype=np.float64)
np_y = np.array(y, dtype=np.float64)
np_z = np.array(z, dtype=np.float64)
np_m = np.array(m, dtype=np.float64)
np_vx = np.array(vx, dtype=np.float64)
np_vy = np.array(vy, dtype=np.float64)
np_vz = np.array(vz, dtype=np.float64)

In [214]:
def calcularForcas(np_x, np_y, np_z, np_m, G, epsilon):
    dx = np_x[None, :] - np_x[:, None]
    dy = np_y[None, :] - np_y[:, None]
    dz = np_z[None, :] - np_z[:, None]

    dist = np.sqrt(dx**2 + dy**2 + dz**2 + epsilon**2)

    np.fill_diagonal(dist, np.inf)

    F = G * (np_m[:, None] * np_m[None, :]) / (dist**2)

    fx = F * (dx / dist)
    fy = F * (dy / dist)
    fz = F * (dz / dist)

    fx_total = np.sum(fx, axis=1)
    fy_total = np.sum(fy, axis=1)
    fz_total = np.sum(fz, axis=1)

    return fx_total, fy_total, fz_total

In [215]:
def atualizarPosicao(np_x, np_y, np_z, np_vx, np_vy, np_vz, np_m, dt, G, epsilon):
    fx, fy, fz = calcularForcas(np_x, np_y, np_z, np_m, G, epsilon)

    ax = fx / np_m
    ay = fy / np_m
    az = fz / np_m

    np_vx += ax * dt
    np_vy += ay * dt
    np_vz += az * dt

    np_x += np_vx * dt
    np_y += np_vy * dt
    np_z += np_vz * dt

In [216]:
def calcularEnergiaCinetica(np_vx, np_vy, np_vz, np_m):
    return 0.5 * np.sum(np_m * (np_vx**2 + np_vy**2 + np_vz**2))

In [217]:
def calcularEnergiaPotencial(np_x, np_y, np_z, np_m, G, epsilon):
    dx = np_x[None, :] - np_x[: ,None]
    dy = np_y[None, :] - np_y[: ,None]
    dz = np_z[None, :] - np_z[: ,None]

    dist = np.sqrt(dx**2 + dy**2 + dz**2 + epsilon**2)

    np.fill_diagonal(dist, np.inf)

    m_prod = np_m[:, None] * np_m[None, :]

    U = G * np.sum(m_prod / dist) / 2

    return U

---

Primeira forma de animação:

* Cores em relação a velocidade

In [160]:
snapshots = np.empty([passos * numCorpos, 8], dtype=np.float64) # inicializando as snapshots vazias para depois serem preenchidas

for i in range(passos): # Simulação e inserção dos resultados de cada passo nas snapshots
    inicio = i * numCorpos # "Index" inicial pra esse passo
    fim = inicio + numCorpos # "Index" final pra esse passo

    snapshots[inicio:fim, 0:8] = np.array([np_x.copy(),
                                           np_y.copy(),
                                           np_z.copy(),
                                           np_vx.copy(),
                                           np_vy.copy(),
                                           np_vz.copy(),
                                           np_m.copy(),
                                           np.full(numCorpos, i)]).transpose() # Salva os resultados atuais no snapshots

    atualizarPosicao(np_x, np_y, np_z, np_vx, np_vy, np_vz, np_m, dt, G, epsilon) # Simula

df = pd.DataFrame(snapshots, columns=["x", "y", "z", "vx", "vy", "vz", "m", "passo"]) # Transforma o snapshots em uma DataFrame

salvarSnapshot(df, biblioteca="Numpy", dimensao="D3") # Salva o DataFrame com a função salvarSnapshot

df = pd.read_hdf("SnapshotTeste.h5") # Lê o arquivo

posicoes = df[["x", "y", "z"]].values  # Pega as posições

velocidades = df[["vx", "vy", "vz"]].values # Pega as velocidade

fig, ax = plt.subplots(subplot_kw={"projection": "3d"}, figsize=(5, 5)) # Cria a figura e os pontos

ax.set_aspect("equal") # Evita que haja distorção entre os eixos x e y

plt.axis("off") # Remove os eixos

fig.set_facecolor("k") # Pinta o fundo da figura de preto

ax.set_facecolor("k")

velocidades = np.sqrt(velocidades[:, 0]**2 + velocidades[:, 1]**2 + velocidades[:, 2]**2) # Calcula a velocidade de cada ponto

scat = ax.scatter([], [], [], c=[], cmap="bwr", s=0.8, vmin=velocidades.min(), vmax=velocidades.max()) # Função que insere os pontos

def update(frame): # Função update
    inicio = frame * numCorpos
    fim = inicio + numCorpos

    pos = posicoes[inicio:fim]
    vel = velocidades[inicio:fim]

    x = pos[:, 0]
    y = pos[:, 1]
    z = pos[:, 2]

    scat.set_array(vel)
    scat._offsets3d = (x, y, z)

    ax.set_xlim3d(x.min() * 1.1, x.max() * 1.1)
    ax.set_ylim3d(y.min() * 1.1, y.max() * 1.1)
    ax.set_zlim3d(z.min() * 1.1, z.max() * 1.1)

    return scat,

anim = animation.FuncAnimation(fig, update, frames=passos, blit=False)

print("Renderizando vídeo...")

writer = animation.FFMpegWriter(fps=20, codec='libx264', bitrate=2000) # Define como a animação vai ser lida e como vai ser salva (20 fps, em .mp4)

anim.save("simulacao_corpos_3D.mp4", writer=writer) # Salva o animação um arquivo .mp4

plt.close() # Evita o plot estático inicial

print("O arquivo foi salvo com sucesso.")

Renderizando vídeo...
O arquivo foi salvo com sucesso.


Segunda forma de animação:

* Cores em relação a massa;
* Tamanhos em relação a massa; <-- Essa parte ainda não foi implementada

In [218]:
cores = ["yellow", "green", "orange", "blue", "red", "brown", "white", "pink", "purple"]

In [219]:
snapshots = np.empty([passos * numCorpos, 8], dtype=np.float64) # inicializando as snapshots vazias para depois serem preenchidas

for i in range(passos): # Simulação e inserção dos resultados de cada passo nas snapshots
    inicio = i * numCorpos # "Index" inicial pra esse passo
    fim = inicio + numCorpos # "Index" final pra esse passo

    snapshots[inicio:fim, 0:8] = np.array([np_x.copy(),
                                           np_y.copy(),
                                           np_z.copy(),
                                           np_vx.copy(),
                                           np_vy.copy(),
                                           np_vz.copy(),
                                           np_m.copy(),
                                           np.full(numCorpos, i)]).transpose() # Salva os resultados atuais no snapshots

    atualizarPosicao(np_x, np_y, np_z, np_vx, np_vy, np_vz, np_m, dt, G, epsilon) # Simula

df = pd.DataFrame(snapshots, columns=["x", "y", "z", "vx", "vy", "vz", "m", "passo"]) # Transforma o snapshots em uma DataFrame

salvarSnapshot(df, biblioteca="Numpy", dimensao="D3") # Salva o DataFrame com a função salvarSnapshot

df = pd.read_hdf("SnapshotTeste.h5") # Lê o arquivo

posicoes = df[["x", "y", "z"]].values  # Pega as posições

fig, ax = plt.subplots(subplot_kw={"projection": "3d"}, figsize=(5, 5)) # Cria a figura e os pontos

ax.set_aspect("equal") # Evita que haja distorção entre os eixos x e y

plt.axis("off") # Remove os eixos

fig.set_facecolor("k") # Pinta o fundo da figura de preto

ax.set_facecolor("k")

scat = ax.scatter([], [], [], c=[], s=0.8) # Função que insere os pontos

def update(frame): # Função update
    inicio = frame * numCorpos
    fim = inicio + numCorpos

    pos = posicoes[inicio:fim]

    x = pos[:, 0]
    y = pos[:, 1]
    z = pos[:, 2]

    scat._offsets3d = (x, y, z)
    scat.set_color(cores)

    ax.set_xlim3d(x.min() * 1.1, x.max() * 1.1)
    ax.set_ylim3d(y.min() * 1.1, y.max() * 1.1)
    ax.set_zlim3d(z.min() * 1.1, z.max() * 1.1)

    return scat,

anim = animation.FuncAnimation(fig, update, frames=passos, blit=False)

print("Renderizando vídeo...")

writer = animation.FFMpegWriter(fps=20, codec='libx264', bitrate=2000) # Define como a animação vai ser lida e como vai ser salva (20 fps, em .mp4)

anim.save("simulacao_corpos_3D.mp4", writer=writer) # Salva o animação um arquivo .mp4

plt.close() # Evita o plot estático inicial

print("O arquivo foi salvo com sucesso.")

Renderizando vídeo...
O arquivo foi salvo com sucesso.
